# HumorVibes — Jestry Live Portal (thin launcher)

Mounts the `punchline-mesh-src` dataset, boots the **Jestry** verified laugh-reuse portal (stdlib HTTP, no pip installs), and exposes it publicly through a Cloudflare quick tunnel. The `*.trycloudflare.com` URL prints below and is announced to ntfy.sh while the session runs (~8h).

Inside Kaggle the S/R/E/B instrument is the attached **Gemma** checkpoint (true teacher-forced log-probabilities). The been-done precedent index runs on the offline hash backend here and labels itself accordingly — semantic embeddinggemma search is the local configuration.

Charter + code: JESTRY-CHARTER-AND-CONSTITUTION-2026-07-23.md in the mounted source.

In [ ]:
import glob, os, re, shutil, subprocess, sys, time, urllib.request

# mount layouts vary (zip-mode datasets can nest); find the source root
# by its marker file instead of assuming the path
hits = sorted(glob.glob('/kaggle/input/**/jestry_portal.py', recursive=True))
assert hits, ('punchline-mesh-src source not found under /kaggle/input: '
              + repr(sorted(glob.glob('/kaggle/input/*/*'))[:40]))
SRC = os.path.dirname(hits[0])
WORK = '/kaggle/working/src'
shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(SRC, WORK)
os.chdir(WORK)
os.environ['GEMMA_PROVIDER'] = 'transformers'   # attached checkpoint = real logprobs
os.environ['JESTRY_PORTAL_PORT'] = '8081'
print('source mounted at', WORK)


In [ ]:
# --- in-kernel instrument calibration (background) ---
# certifies the attached transformers Gemma for acceptance decisions;
# jestry picks the instrument-keyed receipt up automatically when done
cal_proc = subprocess.Popen([sys.executable, 'calibrate_gemma4.py',
                             '--instrument', 'kaggle'],
                            stdout=open('/kaggle/working/calibration.log', 'w'),
                            stderr=subprocess.STDOUT, text=True)
print('calibration launched in background (see calibration.log)')


In [ ]:
# --- boot the stdlib portal (no pip installs) ---
portal_proc = subprocess.Popen([sys.executable, 'jestry_portal.py'],
                               stdout=open('/kaggle/working/portal.log', 'w'),
                               stderr=subprocess.STDOUT, text=True)
deadline = time.time() + 90
ok = False
while time.time() < deadline and not ok:
    try:
        urllib.request.urlopen('http://127.0.0.1:8081/api/charter', timeout=3)
        ok = True
    except Exception:
        time.sleep(1.5)
assert ok, 'portal did not answer: ' + open('/kaggle/working/portal.log').read()[-2000:]
print('portal is answering on 127.0.0.1:8081')


In [ ]:
# --- cloudflared quick tunnel ---
CF = '/kaggle/working/cloudflared'
urllib.request.urlretrieve(
    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', CF)
os.chmod(CF, 0o755)
def start_tunnel():
    return subprocess.Popen([CF, 'tunnel', '--url', 'http://127.0.0.1:8081', '--no-autoupdate'],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
def read_url(proc, timeout=120):
    t0, buf = time.time(), []
    while time.time() - t0 < timeout:
        line = proc.stdout.readline()
        if not line:
            time.sleep(0.2); continue
        buf.append(line)
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m:
            return m.group(0), buf
    return None, buf
cf_proc = start_tunnel()
url, buf = read_url(cf_proc)
assert url, 'no tunnel URL: ' + ''.join(buf)[-2000:]
TOPIC = 'jestry-portal-live-m3w8zr52'
def announce(u):
    try:
        req = urllib.request.Request(f'https://ntfy.sh/{TOPIC}', data=u.encode(),
                                     headers={'Title': 'jestry portal live'})
        urllib.request.urlopen(req, timeout=10)
    except Exception as e:
        print('announce failed:', e)
announce(url)
print('=' * 70)
print('JESTRY PORTAL URL:', url)
print('=' * 70)


In [ ]:
# --- keep-alive ~8h, re-announce hourly, restart tunnel if it drops, exit clean ---
END = time.time() + 8 * 3600
last_announce = time.time()
while time.time() < END:
    time.sleep(30)
    if portal_proc.poll() is not None:
        print('portal died; tail:')
        print(open('/kaggle/working/portal.log').read()[-2000:])
        break
    if cf_proc.poll() is not None:
        print('tunnel died; restarting')
        cf_proc = start_tunnel()
        url2, _ = read_url(cf_proc)
        if url2:
            url = url2; announce(url); print('new URL:', url)
    if time.time() - last_announce > 3600:
        announce(url); last_announce = time.time()
print('session end; final URL was:', url)
for p in (cf_proc, portal_proc):
    try:
        p.terminate()
    except Exception:
        pass
